# Bilig WorkPaper Formula Readback With Haystack

This cookbook shows how to expose spreadsheet-backed business logic to Haystack without opening Excel, Google Sheets, or a browser. The example writes one input cell in a Bilig WorkPaper forecast model, recalculates dependent formulas, verifies readback, and returns a compact proof object.

The first path uses a Haystack `Tool`. The second path wraps the same operation as a reusable Haystack pipeline component.

## Install Dependencies

In [ ]:
! pip install -q -U haystack-ai

## Define The WorkPaper Call

The helper below calls the public Bilig demo API. If the response is not verified, it raises instead of letting an agent or pipeline continue with stale formula values.

In [ ]:
from __future__ import annotations

import json
import os
from typing import Any
from urllib.parse import urljoin
from urllib.request import Request, urlopen

from haystack import Pipeline, component
from haystack.tools import Tool


DEFAULT_BASE_URL = "https://bilig.proompteng.ai"


def call_bilig_forecast(
    sheet_name: str = "Inputs",
    address: str = "B3",
    value: float = 0.4,
    base_url: str | None = None,
    timeout: int = 30,
) -> dict[str, Any]:
    resolved_base_url = base_url or os.getenv("BILIG_BASE_URL", DEFAULT_BASE_URL)
    if not resolved_base_url.startswith(("http://", "https://")):
        raise ValueError("BILIG_BASE_URL must start with http:// or https://")

    endpoint = urljoin(resolved_base_url.rstrip("/") + "/", "api/workpaper/n8n/forecast")
    body = json.dumps({"sheetName": sheet_name, "address": address.upper(), "value": value}).encode("utf-8")
    request = Request(
        endpoint,
        data=body,
        headers={
            "accept": "application/json",
            "content-type": "application/json",
            "user-agent": "Haystack-Bilig-WorkPaper-Cookbook/0.1",
        },
        method="POST",
    )

    with urlopen(request, timeout=timeout) as response:
        parsed = json.loads(response.read().decode("utf-8"))

    if not isinstance(parsed, dict):
        raise ValueError(f"Expected JSON object response, received {type(parsed).__name__}")
    if parsed.get("verified") is not True:
        raise ValueError(f"Bilig returned an unverified WorkPaper response: {parsed}")

    return compact_proof(parsed)


def compact_proof(proof: dict[str, Any]) -> dict[str, Any]:
    before = proof.get("before") if isinstance(proof.get("before"), dict) else {}
    after = proof.get("after") if isinstance(proof.get("after"), dict) else {}
    checks = proof.get("checks") if isinstance(proof.get("checks"), dict) else {}

    return {
        "verified": proof.get("verified") is True,
        "editedCell": proof.get("editedCell"),
        "before": {"expectedArr": before.get("expectedArr"), "targetGap": before.get("targetGap")},
        "after": {"expectedArr": after.get("expectedArr"), "targetGap": after.get("targetGap")},
        "checks": {
            "formulasPersisted": checks.get("formulasPersisted") is True,
            "restoredMatchesAfter": checks.get("restoredMatchesAfter") is True,
            "computedOutputChanged": checks.get("computedOutputChanged") is True,
        },
        "source": "Bilig WorkPaper",
        "github": "https://github.com/proompteng/bilig",
    }

## Use It As A Haystack Tool

This tool can be passed to Haystack agent workflows that support tool/function calling. The direct `invoke()` call below is deterministic and does not require an LLM key.

In [ ]:
bilig_workpaper_tool = Tool(
    name="bilig_workpaper_formula_readback",
    description=(
        "Edit one Bilig WorkPaper forecast input, recalculate formulas, "
        "and return a verified before/after readback proof."
    ),
    parameters={
        "type": "object",
        "properties": {
            "sheet_name": {"type": "string", "default": "Inputs"},
            "address": {"type": "string", "default": "B3"},
            "value": {"type": "number", "default": 0.4},
        },
        "required": ["sheet_name", "address", "value"],
    },
    function=call_bilig_forecast,
)

proof = bilig_workpaper_tool.invoke(sheet_name="Inputs", address="B3", value=0.4)
print(json.dumps(proof, indent=2))

## Use It As A Pipeline Component

The same verified operation can also run inside a Haystack pipeline when the workbook calculation is part of a larger deterministic workflow.

In [ ]:
@component
class BiligWorkPaperFormulaReadback:
    @component.output_types(proof=dict)
    def run(self, sheet_name: str = "Inputs", address: str = "B3", value: float = 0.4) -> dict[str, dict[str, Any]]:
        return {"proof": call_bilig_forecast(sheet_name=sheet_name, address=address, value=value)}


pipeline = Pipeline()
pipeline.add_component("workpaper", BiligWorkPaperFormulaReadback())

result = pipeline.run({"workpaper": {"sheet_name": "Inputs", "address": "B3", "value": 0.4}})
result["workpaper"]["proof"]

## Self-Hosted Bilig

Set `BILIG_BASE_URL` if you run Bilig yourself:

```bash
export BILIG_BASE_URL=http://localhost:4321
```

The cookbook uses `POST /api/workpaper/n8n/forecast` with `sheetName`, `address`, and `value` in the JSON payload.